In [23]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

In [24]:
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: str | ForwardRef('os.PathLike[str]') | None = None, stream: IO[str] | None = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: str | None = 'utf-8') -> bool>

In [25]:
model = ChatGroq(model="openai/gpt-oss-20b")

In [26]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [27]:
def generate_joke(state: JokeState) :
    prompt = f'generate a joke on the topic {state["topic"]}'
    response = model.invoke(prompt).content
    return {
        'joke': response,
    }

In [28]:
def generate_explanation(state:JokeState):
    prompt = f'explain the joke: {state["joke"]}'
    response = model.invoke(prompt).content
    return {
        'explanation': response,
    }

In [29]:
graph = StateGraph(JokeState)
graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)
graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer)


In [30]:
config1 = {"configurable": {"thread_id":"1"}}
workflow.invoke({"topic": "programming"}, config=config1)

{'topic': 'programming',
 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs!',
 'explanation': '**The joke is a pun that relies on two very different meanings of the word “bug.”**\n\n| Meaning | Context |\n|---------|---------|\n| **Bugs as insects** | In nature, many insects (flies, moths, beetles, etc.) are drawn to light sources. When you turn on a lamp or a screen, the insects are attracted to it. |\n| **Bugs as software glitches** | In programming, a “bug” is a defect or error in the code that causes it to behave incorrectly. |\n\nThe joke goes:  \n\n> **“Why do programmers prefer dark mode? Because light attracts bugs!”**\n\n**What’s happening?**\n\n1. **Literal interpretation** – If you have a bright screen (light mode), you’ll actually attract real insects, just like a street lamp does at night.  \n2. **Metaphorical interpretation** – In the world of coding, “light mode” is a bright‑white background for your editor. The punchline suggests that a brigh